# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**

<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">

## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label


In [1]:
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [ ]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


## 2. Preprocessing the Dataset

In [3]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# Apply preprocessing to all reviews
df["review"] = df["review"].apply(preprocess_text)

df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


## 3. Split the Data

We divide the dataset into:

- **80% training data** — used to train the model
- **20% testing data** — used to evaluate the model

`stratify=y` keeps the proportion of positive and negative reviews similar in both sets.

In [5]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training reviews: 800
Testing reviews: 200

Training labels:
label
1    400
0    400
Name: count, dtype: int64

Testing labels:
label
1    100
0    100
Name: count, dtype: int64


## 4. Convert Text to TF-IDF Vectors

A neural network cannot directly understand text.

**TF-IDF converts each review into a vector of numbers.**


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


## 5. Convert the Data to PyTorch Tensors

PyTorch neural networks work with **tensors**.

In [7]:
import torch
import torch.nn as nn

# Convert TF-IDF data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_tfidf.toarray())
X_test_tensor = torch.FloatTensor(X_test_tfidf.toarray())

y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

## 6. Build a Simple Neural Network

Our network has:

1. **Input layer** — receives the TF-IDF features
2. **Hidden layer** — learns useful patterns
3. **ReLU activation** — adds non-linearity
4. **Output layer** — produces one value for binary classification

We do **not** put `Sigmoid` inside the model because `BCEWithLogitsLoss` applies it internally in a numerically stable way.

In [8]:
input_size = X_train_tensor.shape[1]
# torch.size([800, 1000])

model = nn.Sequential(
    nn.Linear(input_size, 64),   # Input layer
    nn.ReLU(),

    nn.Linear(64, 32),           # Hidden layer
    nn.ReLU(),

    nn.Linear(32, 1)             # Output layer
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


## 7. Choose the Loss Function and Optimizer

- **BCEWithLogitsLoss** is used for binary classification.
To measure the model’s mistakes.
- **Adam** is responsible for changing the model's weights so that the loss becomes smaller.

In [9]:
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 8. Train the Neural Network

One **epoch** means the model has seen the full training dataset once.

For every epoch:

1. Make predictions
2. Calculate the loss
3. Calculate gradients with backpropagation
4. Update the weights

In [10]:
epochs = 10

for epoch in range(epochs):
    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}")

Epoch 5/10 - Loss: 0.6423
Epoch 10/10 - Loss: 0.4368


## 9. Evaluate the Model

The model outputs raw scores called **logits**.

We apply `sigmoid` to convert them to probabilities between 0 and 1.

- Probability **≥ 0.5 → Positive (1)**
- Probability **< 0.5 → Negative (0)**

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test = model(X_test_tensor)
    test_probabilities = torch.sigmoid(test)
    predictions = (test_probabilities >= 0.5).int().numpy().flatten()

accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.750


## 10. Classification Report and Confusion Matrix

The classification report shows:

- **Precision**
- **Recall**
- **F1-score**

The confusion matrix shows how many reviews were classified correctly and incorrectly.

In [12]:
print("Classification Report:")
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

    Negative       0.75      0.76      0.75       100
    Positive       0.76      0.74      0.75       100

    accuracy                           0.75       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.75      0.75      0.75       200

Confusion Matrix:
[[76 24]
 [26 74]]


## Task 1 – Represent Reviews Using Word2Vec

In the previous part of the lab, TF-IDF was used to convert each review into numerical features.
Now, use Word2Vec instead.

Requirements:
Tokenize the training and testing reviews.
Train a Word2Vec model using the training reviews.
Use:
vector_size = 200
window = 5
min_count = 1
sg = 1

Convert each review into a single vector.
Convert the resulting vectors into PyTorch tensors.


In [13]:
# Tokenize the training and testing reviews

train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

print("First training review:")
print(X_train.iloc[0])

print("\nTokenized version:")
print(train_tokens[0])

print("\nNumber of training reviews:", len(train_tokens))
print("Number of testing reviews:", len(test_tokens))

First training review:
it is perfect for a sitdown family meal or get together with a few friends

Tokenized version:
['it', 'is', 'perfect', 'for', 'a', 'sitdown', 'family', 'meal', 'or', 'get', 'together', 'with', 'a', 'few', 'friends']

Number of training reviews: 800
Number of testing reviews: 200


In [16]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 45.7 MB/s eta 0:00:00


In [26]:
from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=100
)

print("Word2Vec model retrained successfully!")
print("Vector size:", word2vec_model.vector_size)
print("Vocabulary size:", len(word2vec_model.wv.index_to_key))

Word2Vec model retrained successfully!
Vector size: 200
Vocabulary size: 1787


In [27]:
import numpy as np

def review_to_vector(tokens, model):
    vectors = []

    for word in tokens:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


# Convert all training reviews to vectors
X_train_w2v = np.array([
    review_to_vector(tokens, word2vec_model)
    for tokens in train_tokens
])

# Convert all testing reviews to vectors
X_test_w2v = np.array([
    review_to_vector(tokens, word2vec_model)
    for tokens in test_tokens
])

print("Training Word2Vec shape:", X_train_w2v.shape)
print("Testing Word2Vec shape:", X_test_w2v.shape)

Training Word2Vec shape: (800, 200)
Testing Word2Vec shape: (200, 200)


In [28]:
import torch

# Convert Word2Vec features to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_w2v)
X_test_tensor = torch.FloatTensor(X_test_w2v)

# Convert labels to PyTorch tensors
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)

print("X_train_tensor shape:", X_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)

print("y_train_tensor shape:", y_train_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([800, 200])
X_test_tensor shape: torch.Size([200, 200])
y_train_tensor shape: torch.Size([800, 1])
y_test_tensor shape: torch.Size([200, 1])


## Task 2 – Build a Different Neural Network

Use the Word2Vec vectors from Task 1 as the input to a new neural network.
Your network must contain:
An input layer
Three hidden layers
ReLU activation functions
One output neuron
Use the following architecture:

Input
   
   ↓

128 neurons
   
   ↓

64 neurons
   
   ↓

32 neurons
   
   ↓

1 output neuron

Train the model using:
, BCEWithLogitsLoss
, Adam optimizer
, Learning rate = 0.001
, 15 epochs.
Then calculate the test accuracy.

In [29]:
import torch.nn as nn

class SentimentNN(nn.Module):
    def __init__(self):
        super(SentimentNN, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(200, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


# Create the model
model = SentimentNN()

print(model)

SentimentNN(
  (network): Sequential(
    (0): Linear(in_features=200, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [30]:
import torch.optim as optim

# Loss function
criterion = nn.BCEWithLogitsLoss()

# Adam optimizer with learning rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer: Adam")
print("Learning rate:", optimizer.param_groups[0]['lr'])

Loss function: BCEWithLogitsLoss()
Optimizer: Adam
Learning rate: 0.001


In [31]:
# Train the neural network for 15 epochs

num_epochs = 15

for epoch in range(num_epochs):

    # Put model in training mode
    model.train()

    # Clear old gradients
    optimizer.zero_grad()

    # Make predictions
    outputs = model(X_train_tensor)

    # Calculate the error
    loss = criterion(outputs, y_train_tensor)

    # Backpropagation
    loss.backward()

    # Update the model weights
    optimizer.step()

    print(f"Epoch {epoch+1}/15, Loss: {loss.item():.4f}")

Epoch 1/15, Loss: 0.6954
Epoch 2/15, Loss: 0.6944
Epoch 3/15, Loss: 0.6935
Epoch 4/15, Loss: 0.6926
Epoch 5/15, Loss: 0.6917
Epoch 6/15, Loss: 0.6906
Epoch 7/15, Loss: 0.6895
Epoch 8/15, Loss: 0.6882
Epoch 9/15, Loss: 0.6868
Epoch 10/15, Loss: 0.6852
Epoch 11/15, Loss: 0.6835
Epoch 12/15, Loss: 0.6816
Epoch 13/15, Loss: 0.6795
Epoch 14/15, Loss: 0.6771
Epoch 15/15, Loss: 0.6743


In [32]:
# Evaluate the model on the test set

model.eval()

with torch.no_grad():

    # Get raw model outputs
    test_outputs = model(X_test_tensor)

    # Convert logits to probabilities
    test_probabilities = torch.sigmoid(test_outputs)

    # Convert probabilities to 0 or 1
    test_predictions = (test_probabilities >= 0.5).float()

    # Calculate accuracy
    correct = (test_predictions == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)

    accuracy = correct / total

print("Correct predictions:", correct)
print("Total test reviews:", total)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Correct predictions: 123
Total test reviews: 200
Test Accuracy: 61.50%
